In [1]:
!pip install transformers sentencepiece

In [2]:
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")
model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base").to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [5]:
user_input = input("You: ").strip()

if user_input == "":
    print("Chatbot: Please say something.")
else:
    print("You entered:", user_input)

You: hlo
You entered: hlo


In [6]:
def generate_response(user_input):
    prompt = f"Answer briefly: {user_input}"

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        temperature=0.5,
        do_sample=True
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Remove prompt if repeated
    if response.startswith(prompt):
        response = response[len(prompt):].strip()

    return response

In [7]:
memory = {}

print("Chatbot: Hello! I am your AI assistant. Type 'exit' to end the chat.\n")

while True:
    user_input = input("You: ").strip()
    user_lower = user_input.lower()

    # Handle empty input
    if user_input == "":
        print("Chatbot: Please say something.")
        continue
    if user_lower in ['exit', 'quit']:
      print("Chatbot: Goodbye! Have a great day!")
      break

Chatbot: Hello! I am your AI assistant. Type 'exit' to end the chat.

You: what is artificial intelligence?
You: exit
Chatbot: Goodbye! Have a great day!


In [8]:
memory = {}

print("Chatbot: Hello! I am your AI assistant. Type 'exit' to end the chat.\n")

def generate_response(user_input):
    prompt = f"Answer briefly: {user_input}"

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        temperature=0.5,
        do_sample=True
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if response.startswith(prompt):
        response = response[len(prompt):].strip()

    return response


while True:
    user_input = input("You: ").strip()
    user_lower = user_input.lower()

    # 🔹 Exit
    if user_lower in ['exit', 'quit']:
        print("Chatbot: Goodbye! Have a great day!")
        break

    # 🔹 Empty input
    if user_input == "":
        print("Chatbot: Please say something.")
        continue

    # 🔹 Greetings
    if user_lower in ["hello", "hi", "hey"]:
        print("Chatbot: Hello! Nice to meet you. How can I assist you today?")
        continue

    # 🔹 Memory
    if "my name is" in user_lower:
        name = user_input.split("my name is")[-1].strip()
        memory["name"] = name
        print(f"Chatbot: Nice to meet you, {name}!")
        continue

    if "what is my name" in user_lower:
        if "name" in memory:
            print(f"Chatbot: Your name is {memory['name']}.")
        else:
            print("Chatbot: I don't know your name yet.")
        continue

    # 🔹 Polite replies
    if "thank you" in user_lower:
        print("Chatbot: You're welcome! Feel free to ask more questions.")
        continue

    if "artificial intelligence" in user_lower or user_lower == "what is ai":
        print("Chatbot: Artificial Intelligence refers to the simulation of human intelligence by machines that can perform tasks such as learning, reasoning, and problem solving.")
        continue

    if "who invented python" in user_lower or "who created python" in user_lower:
        print("Chatbot: Python was created by Guido van Rossum and released in 1991.")
        continue

    # 🔹 Model fallback
    if "what is" in user_lower or "who is" in user_lower or "explain" in user_lower:
        response = generate_response(user_input)
        print("Chatbot:", response)
    else:
        print("Chatbot: I'm not sure how to respond. Please ask a question.")

Chatbot: Hello! I am your AI assistant. Type 'exit' to end the chat.

You: what is artificial intelligence?
Chatbot: Artificial Intelligence refers to the simulation of human intelligence by machines that can perform tasks such as learning, reasoning, and problem solving.
You: who created python?
Chatbot: Python was created by Guido van Rossum and released in 1991.
You: exit
Chatbot: Goodbye! Have a great day!
